# Adding derived variables to the model

**Variables:**
- low disposable income (Binary)
- Log address tenure (Cont) - Stability

In [2]:
# Opening the datasets

from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
)

# Locate the project folder.
relative_folder = Path("data") / "KGB Modelling"

project_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_folder / "KGB in sample.csv").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the KGB modelling data.")

data_folder = project_root / relative_folder

# Load the development and validation samples.
kgb_in_sample = pd.read_csv(data_folder / "KGB in sample.csv")
kgb_out_sample = pd.read_csv(data_folder / "KGB out sample.csv")

target = "target_bad"

# Check the target is complete and binary.
for name, data in [
    ("In-sample", kgb_in_sample),
    ("Out-of-sample", kgb_out_sample),
]:
    if target not in data.columns:
        raise ValueError(f"{name}: missing {target}.")

    if not data[target].isin([0, 1]).all():
        raise ValueError(f"{name}: {target} must contain only 0 and 1.")

    if data[target].nunique() != 2:
        raise ValueError(f"{name}: both good and bad accounts are required.")

# Review sample sizes and bad rates.
sample_summary = pd.DataFrame([
    {
        "Sample": name,
        "Accounts": len(data),
        "Bads": int(data[target].sum()),
        "Bad rate": data[target].mean(),
    }
    for name, data in [
        ("In-sample", kgb_in_sample),
        ("Out-of-sample", kgb_out_sample),
    ]
])

In [4]:
# Creating the variables

def add_derived_variables(data):
    data = data.copy()

    required_columns = [
        "disposable_income",
        "time_at_address_months",
        "employment_tenure_months",
    ]

    missing_columns = set(required_columns) - set(data.columns)
    if missing_columns:
        raise ValueError(f"Missing columns: {sorted(missing_columns)}")

    for column in required_columns:
        data[column] = pd.to_numeric(data[column], errors="raise")

        if data[column].isin([np.inf, -np.inf]).any():
            raise ValueError(f"{column} contains infinite values.")

    for column in [
        "time_at_address_months",
        "employment_tenure_months",
    ]:
        if data[column].dropna().lt(0).any():
            raise ValueError(f"{column} contains negative tenure.")

    # Nullable numbers preserve missing inputs in comparisons.
    surplus = data["disposable_income"].astype("Float64")
    address_tenure = data["time_at_address_months"].astype("Float64")
    employment_tenure = data["employment_tenure_months"].astype("Float64")

    # 1 = disposable income below £150; 0 = £150 or above.
    data["low_disposable_income_flag"] = surplus.lt(150).astype("Int64")

    # Natural logarithm; zero months becomes zero.
    data["log_address_tenure"] = np.log1p(address_tenure)

    # Count of employment/address tenures below 12 months.
    data["employment_address_instability_count"] = (
        employment_tenure.lt(12).astype("Int64")
        + address_tenure.lt(12).astype("Int64")
    )

    return data


# Apply identical definitions to both samples.
kgb_in_sample = add_derived_variables(kgb_in_sample)
kgb_out_sample = add_derived_variables(kgb_out_sample)

derived_columns = [
    "low_disposable_income_flag",
    "log_address_tenure",
    "employment_address_instability_count",
]


In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.metrics import (
    brier_score_loss, log_loss, roc_auc_score, roc_curve
)

# 1. Locate and open the KGB in-sample and out-of-sample data.
relative_dir = Path("data") / "KGB Modelling"
project_root = next(
    (
        folder for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_dir / "KGB in sample.csv").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the KGB sample files.")

data_dir = project_root / relative_dir

numeric_predictors = [
    "committed_monthly_outgoings",
    "debt_to_income_ratio",
    "recent_application_count",
    "revolving_utilisation",
    "historic_defaults",
    "DelphiScore",
]
required = [
    "target_bad", "loan_purpose", *numeric_predictors,
    "disposable_income", "time_at_address_months",
    "employment_tenure_months",
]

def prepare_sample(filename):
    data = pd.read_csv(data_dir / filename)
    rows = data[required].copy().replace(r"^\s*$", np.nan, regex=True)

    for column in set(required) - {"loan_purpose"}:
        rows[column] = pd.to_numeric(rows[column], errors="raise")

    rows = rows.replace([np.inf, -np.inf], np.nan).dropna().copy()
    print(
        f"{filename}: retained {len(rows):,}; "
        f"excluded {len(data) - len(rows):,}"
    )

    if (
        not rows["target_bad"].isin([0, 1]).all()
        or rows["target_bad"].nunique() != 2
    ):
        raise ValueError("Each sample must contain both target classes.")

    tenure = rows[["time_at_address_months", "employment_tenure_months"]]
    if (tenure < 0).any().any():
        raise ValueError("Tenure values cannot be negative.")

    # 2. Create the three derived variables.
    rows["low_disposable_income_flag"] = (
        rows["disposable_income"] < 150
    ).astype(float)

    rows["log_address_tenure"] = np.log1p(
        rows["time_at_address_months"]
    )

    rows["employment_address_instability_count"] = (
        (rows["employment_tenure_months"] < 12).astype(float)
        + (rows["time_at_address_months"] < 12).astype(float)
    )
    return rows

train = prepare_sample("KGB in sample.csv")
test = prepare_sample("KGB out sample.csv")

# Learn scaling from the in-sample data only.
scaled_predictors = [*numeric_predictors, "log_address_tenure"]
train_means = train[scaled_predictors].mean()
train_stds = train[scaled_predictors].std(ddof=0)

if (train_stds == 0).any():
    raise ValueError("A training predictor has no variation.")

def make_design_matrix(rows):
    X = (rows[scaled_predictors] - train_means) / train_stds
    X.insert(0, "const", 1.0)

    for column in [
        "low_disposable_income_flag",
        "employment_address_instability_count",
    ]:
        X[column] = rows[column]

    X["loan_purpose=education"] = (
        rows["loan_purpose"] == "education"
    ).astype(float)
    return X.astype(float)

X_train = make_design_matrix(train)
X_test = make_design_matrix(test)
y_train = train["target_bad"].astype(int)
y_test = test["target_bad"].astype(int)

if np.linalg.matrix_rank(X_train.to_numpy()) < X_train.shape[1]:
    raise ValueError("Training predictors are perfectly collinear.")

# Fit the fixed reduced specification on in-sample accounts only.
reduced_model = sm.Logit(y_train, X_train).fit(
    maxiter=200, disp=False
)
if not reduced_model.mle_retvals.get("converged", False):
    raise RuntimeError("Model did not converge.")
if (
    not np.isfinite(reduced_model.params).all()
    or not np.isfinite(reduced_model.bse).all()
):
    raise RuntimeError("Invalid coefficients or standard errors.")

train_pd = reduced_model.predict(X_train)
test_pd = reduced_model.predict(X_test)

# 3. Display discrimination, probability accuracy and calibration metrics.
def performance(actual, probability):
    auc = roc_auc_score(actual, probability)
    fpr, tpr, _ = roc_curve(actual, probability)

    return {
        "Accounts": len(actual),
        "Bad rate": actual.mean(),
        "Mean predicted PD": probability.mean(),
        "Brier": brier_score_loss(actual, probability),
        "ROC AUC": auc,
        "Gini": 2 * auc - 1,
        "KS": np.max(tpr - fpr),
        "Log loss": log_loss(actual, probability),
    }

results = pd.DataFrame({
    "In-sample": performance(y_train, train_pd),
    "Out-of-sample": performance(y_test, test_pd),
})

display(results.round(4))
print(f"Training AIC: {reduced_model.aic:.2f}")
print(f"Training McFadden pseudo R²: {reduced_model.prsquared:.4f}")
print(reduced_model.summary())

KGB in sample.csv: retained 7,258; excluded 0
KGB out sample.csv: retained 2,458; excluded 0


,In-sample,Out-of-sample
Accounts,7258.0000,2458.0000
Bad rate,0.0405,0.0415
Mean predicted PD,0.0405,0.0393
Brier,0.0385,0.0394
ROC AUC,0.6414,0.6345
Gini,0.2829,0.2690
KS,0.2196,0.2044
Log loss,0.1647,0.1680


Training AIC: 2412.73
Training McFadden pseudo R²: 0.0286
                           Logit Regression Results                           
Dep. Variable:             target_bad   No. Observations:                 7258
Model:                          Logit   Df Residuals:                     7247
Method:                           MLE   Df Model:                           10
Date:                Sun, 27 Sep 2026   Pseudo R-squ.:                 0.02864
Time:                        20:19:19   Log-Likelihood:                -1195.4
converged:                       True   LL-Null:                       -1230.6
Covariance Type:            nonrobust   LLR p-value:                 3.562e-11
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
const                                   -3.3108      0.077    -42.740      0.000      -3.463      -3

---

# Reduced model comparison

In [8]:
# Remove only the two requested predictors.
removed = [
    "DelphiScore",
    "employment_address_instability_count",
    "committed_monthly_outgoings",
]
X_train_new = X_train.drop(columns=removed)
X_test_new = X_test.drop(columns=removed)

# Refit all remaining coefficients using the same training accounts.
new_model = sm.Logit(y_train, X_train_new).fit(
    maxiter=200, disp=False
)

if not new_model.mle_retvals.get("converged", False):
    raise RuntimeError("Model did not converge.")
if (
    not np.isfinite(new_model.params).all()
    or not np.isfinite(new_model.bse).all()
):
    raise RuntimeError("Invalid coefficients or standard errors.")

new_train_pd = new_model.predict(X_train_new)
new_test_pd = new_model.predict(X_test_new)

comparison = pd.DataFrame({
    "Original in-sample": performance(y_train, train_pd),
    "New in-sample": performance(y_train, new_train_pd),
    "Original out-of-sample": performance(y_test, test_pd),
    "New out-of-sample": performance(y_test, new_test_pd),
})

display(comparison.round(6))
print(f"Original training AIC: {reduced_model.aic:.2f}")
print(f"New training AIC: {new_model.aic:.2f}")
print(f"New training McFadden pseudo R²: {new_model.prsquared:.4f}")
print(new_model.summary())

,Original in-sample,New in-sample,Original out-of-sample,New out-of-sample
Accounts,7258.000000,7258.000000,2458.000000,2458.000000
Bad rate,0.040507,0.040507,0.041497,0.041497
Mean predicted PD,0.040507,0.040507,0.039345,0.039144
Brier,0.038473,0.038501,0.039381,0.039364
ROC AUC,0.641426,0.640718,0.634492,0.644179
Gini,0.282851,0.281436,0.268984,0.288358
KS,0.219599,0.228754,0.204409,0.220147
Log loss,0.164696,0.164899,0.167990,0.167671


Original training AIC: 2412.73
New training AIC: 2409.68
New training McFadden pseudo R²: 0.0274
                           Logit Regression Results                           
Dep. Variable:             target_bad   No. Observations:                 7258
Model:                          Logit   Df Residuals:                     7250
Method:                           MLE   Df Model:                            7
Date:                Sun, 27 Sep 2026   Pseudo R-squ.:                 0.02744
Time:                        20:34:45   Log-Likelihood:                -1196.8
converged:                       True   LL-Null:                       -1230.6
Covariance Type:            nonrobust   LLR p-value:                 4.634e-12
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -3.3139      0.075    -44.195      0.000      -3.46